<a href="https://colab.research.google.com/github/thuynhungp068-design/cuda_learning_with_google_colab/blob/main/cuda_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%%writefile hello.cu

#include <cstdio>

__global__ void hello_kernel(){
  printf("Hello from GPU! block=%d,thread=%d\n",blockIdx.x,threadIdx.x);
}

int main(){
  hello_kernel<<<2,4>>>();
  cudaDeviceSynchronize();
  return 0;
}

Writing hello.cu


In [3]:
!nvcc -o hello hello.cu

In [4]:
!./hello

Hello from GPU! block=1,thread=0
Hello from GPU! block=1,thread=1
Hello from GPU! block=1,thread=2
Hello from GPU! block=1,thread=3
Hello from GPU! block=0,thread=0
Hello from GPU! block=0,thread=1
Hello from GPU! block=0,thread=2
Hello from GPU! block=0,thread=3


In [5]:
%%writefile vector_add.cu

#include <cstdio>

__global__ void vector_add(const float* a,const float* b,float* result,int n){
  int i=blockIdx.x*blockDim.x+threadIdx.x;
  if(i<n){
    result[i]=a[i]+b[i];
  }
}

int main(){
  const int N=1024;
  const size_t size=N*sizeof(float);

  float h_a[N],h_b[N],h_result[N];
  for(int i=0;i<N;i++){
    h_a[i]=(float)i;
    h_b[i]=(float)N-i;
  }

  float *d_a,*d_b,*d_result;
  cudaMalloc(&d_a,size);
  cudaMalloc(&d_b,size);
  cudaMalloc(&d_result,size);

  cudaMemcpy(d_a,h_a,size,cudaMemcpyHostToDevice);
  cudaMemcpy(d_b,h_b,size,cudaMemcpyHostToDevice);

  int threads_per_block=256;
  int blocks=(N+threads_per_block-1)/threads_per_block;
  vector_add<<<blocks,threads_per_block>>>(d_a,d_b,d_result,N);

  cudaDeviceSynchronize();

  cudaMemcpy(h_result,d_result,size,cudaMemcpyDeviceToHost);

  bool correct=true;
  for(int i=0;i<N;i++){
    if(h_result[i]!=N){
      correct=false;
      printf("Error at i=%d: %f\\n",i,h_result[i]);
      break;
    }
  }
  printf("%s\\n", correct ? "Vector add PASSED!" : "Vector add FAILED!");

  cudaFree(d_a);
  cudaFree(d_b);
  cudaFree(d_result);
  return 0;
}

Writing vector_add.cu


In [6]:
!nvcc -arch=sm_75 -o vector_add vector_add.cu

In [7]:
!./vector_add


Vector add PASSED!\n

In [8]:
!nvidia-smi -q | grep -E "SM|Memory|L2|Multiprocessor"

==============NVSMI LOG==============
    FB Memory Usage
    BAR1 Memory Usage
    Conf Compute Protected Memory Usage
        Memory                            : 0 %
        Memory Current Temp               : N/A
        Memory Max Operating Temp         : N/A
    GPU Memory Power Readings 
        SM                                : 1050 MHz
        Memory                            : 5000 MHz
        Memory                            : 5001 MHz
        Memory                            : 5001 MHz
        Memory                            : N/A
        SM                                : 1590 MHz
        Memory                            : 5001 MHz


In [13]:
%%writefile block_mapping.cu

#include <cuda_runtime.h>
#include <stdio.h>

__device__ uint get_smid() {
    uint smid;
    asm volatile("mov.u32 %0, %%smid;" : "=r"(smid));
    return smid;
}

__global__ void show_mapping() {
    if (threadIdx.x == 0) {
        printf("Block (%d,%d) -> SM %d\n",
            blockIdx.x, blockIdx.y, get_smid());
    }
}

int main() {
    show_mapping<<<80, 32>>>();
    cudaDeviceSynchronize();
    return 0;
}

Overwriting block_mapping.cu


In [14]:
!nvcc -arch=sm_75 -o block_mapping block_mapping.cu

In [15]:
!./block_mapping

Block (24,0) -> SM 9
Block (34,0) -> SM 29
Block (14,0) -> SM 28
Block (4,0) -> SM 8
Block (74,0) -> SM 29
Block (29,0) -> SM 19
Block (39,0) -> SM 39
Block (64,0) -> SM 9
Block (54,0) -> SM 28
Block (9,0) -> SM 18
Block (20,0) -> SM 1
Block (19,0) -> SM 38
Block (30,0) -> SM 21
Block (44,0) -> SM 8
Block (0,0) -> SM 0
Block (69,0) -> SM 19
Block (22,0) -> SM 5
Block (10,0) -> SM 20
Block (79,0) -> SM 39
Block (32,0) -> SM 25
Block (60,0) -> SM 1
Block (49,0) -> SM 18
Block (31,0) -> SM 23
Block (2,0) -> SM 4
Block (70,0) -> SM 21
Block (23,0) -> SM 7
Block (59,0) -> SM 38
Block (21,0) -> SM 3
Block (12,0) -> SM 24
Block (25,0) -> SM 11
Block (33,0) -> SM 27
Block (11,0) -> SM 22
Block (62,0) -> SM 5
Block (35,0) -> SM 31
Block (3,0) -> SM 6
Block (1,0) -> SM 2
Block (72,0) -> SM 25
Block (40,0) -> SM 0
Block (13,0) -> SM 26
Block (71,0) -> SM 23
Block (27,0) -> SM 15
Block (50,0) -> SM 20
Block (63,0) -> SM 7
Block (26,0) -> SM 13
Block (37,0) -> SM 35
Block (5,0) -> SM 10
Block (73,0